In [ ]:
# ========================================================
# LANGKAH 1: Instalasi Library Pendukung
# ========================================================
!pip install ultralytics opencv-python matplotlib

# ========================================================
# LANGKAH 2: Kode Pelacakan & Perekaman Snapshot Otomatis
# ========================================================
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# 1. Muat model pre-trained YOLOv8 nano
model = YOLO('yolov8n.pt')

# 2. STRUKTUR DATASET: Silakan sesuaikan nama file asli .mp4 kamu di sini
# Nama label di sebelah kiri akan otomatis menjadi nama file snapshot .png
dataset_videos = {
    "POV_Webcam_User_Aktif": "webcam_ada_orang.mp4",
    "POV_Webcam_Meja_Kosong": "webcam_pc_doang.mp4",
    "POV_CCTV_User_Aktif_Mengecek_Sistem": "cctv_ada_orang.mp4",
    "POV_CCTV_Ruangan_Kosong_Istirahat": "cctv_tanpa_orang.mp4"
}

# Ambang batas waktu simulasi (10 detik untuk keperluan demonstrasi)
BATAS_WAKTU_SIMULASI = 10.0

print("=== MEMULAI PROSES GENERATOR SNAPSHOT LAPORAN K3 ===")

for label_tugas, file_name in dataset_videos.items():

    if not os.path.exists(file_name):
        print(f"\n[PERINGATAN] File '{file_name}' tidak ditemukan. Melewati {label_tugas}...")
        continue

    print(f"\n>> Memproses Analisis & Snapshot untuk: {label_tugas}...")

    cap = cv2.VideoCapture(file_name)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if int(cap.get(cv2.CAP_PROP_FPS)) > 0 else 30
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Simpan output video hasil tracking
    output_video_path = f"output_{file_name}"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    person_time_counter = {}
    snapshot_diambil = False

    # Jalankan tracking ByteTRACK
    results = model.track(source=file_name, persist=True, tracker="bytetrack.yaml", stream=True)

    for frame_idx, r in enumerate(results):
        frame = r.orig_img
        boxes = r.boxes.xyxy.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy()

        status_k3 = "STATUS K3: MEJA KOSONG / USER SEDNG ISTIRAHAT"
        color_text = (255, 255, 255)

        if r.boxes.id is not None:
            track_ids = r.boxes.id.cpu().numpy().astype(int)

            # --- LOGIKA MAIN TARGET FILTER UNTUK POV CCTV ---
            # Jika ada banyak orang di CCTV, kita prioritaskan ID terkecil (User Utama)
            person_ids = [tid for tid, c in zip(track_ids, clss) if int(c) == 0]
            main_target_id = min(person_ids) if person_ids else None

            for box, track_id, cls in zip(boxes, track_ids, clss):
                if int(cls) == 0:
                    x1, y1, x2, y2 = box

                    # Jika ini adalah POV CCTV dan ada orang lain lewat, abaikan orang kedua tersebut
                    #if "CCTV" in label_tugas and track_id != main_target_id:
                        # Tetap beri kotak tipis untuk membuktikan sistem mengenali objek lain tapi tidak menghitung timernya
                        #cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (200, 200, 200), 1)
                        #continue

                    # Hitung waktu hanya untuk target utama
                    if track_id not in person_time_counter:
                        person_time_counter[track_id] = 0
                    person_time_counter[track_id] += 1

                    duration_seconds = person_time_counter[track_id] / fps

                    # Penentuan kondisi warna box (Hijau = Aman, Merah = Terlalu Lama Duduk)
                    if duration_seconds > BATAS_WAKTU_SIMULASI:
                        status_k3 = f"PERINGATAN K3 ({label_tugas.split('_')[1]}): USER ID {track_id} TERLALU LAMA! ({duration_seconds:.1f}s)"
                        color_text = (0, 0, 255)
                        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 4)
                    else:
                        status_k3 = f"STATUS K3 ({label_tugas.split('_')[1]}): USER ID {track_id} AKTIF ({duration_seconds:.1f}s)"
                        color_text = (0, 255, 0)
                        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

                    cv2.putText(frame, f"Person ID:{track_id} [{duration_seconds:.1f}s]", (int(x1), int(y1) - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_text, 2)

        # Gambar teks status utama di pojok kiri atas
        cv2.putText(frame, status_k3, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_text, 2)
        out.write(frame)

        # AMBIL SNAPSHOT SECARA OTOMATIS
        # Diambil pada frame ke-120 (sekitar detik ke-4 setelah video berjalan) agar simulasi waktunya terlihat mantap
        target_snapshot_frame = min(120, total_frames - 5)
        if frame_idx == target_snapshot_frame and not snapshot_diambil:
            snapshot_name = f"snapshot_{label_tugas}.png"
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"Hasil Ekstraksi Laporan: {label_tugas.replace('_', ' ')}", fontsize=10)
            plt.axis('off')
            plt.savefig(snapshot_name, bbox_inches='tight', dpi=150)
            plt.close()
            snapshot_diambil = True

    cap.release()
    out.release()
    print(f"✓ File snapshot sukses dibuat: snapshot_{label_tugas}.png")

print("\n=== SEMUA SNAPSHOT POV WEBCAM & CCTV SIAP DIGUNAKAN ===")

=== MEMULAI PROSES GENERATOR SNAPSHOT LAPORAN K3 ===

>> Memproses Analisis & Snapshot untuk: POV_Webcam_User_Aktif...

video 1/1 (frame 1/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 184.6ms
video 1/1 (frame 2/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 158.6ms
video 1/1 (frame 3/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 145.0ms
video 1/1 (frame 4/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 149.9ms
video 1/1 (frame 5/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 146.2ms
video 1/1 (frame 6/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 151.8ms
video 1/1 (frame 7/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 131.8ms
video 1/1 (frame 8/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 136.1ms
video 1/1 (frame 9/489) /content/webcam_ada_orang.mp4: 384x640 1 person, 7 chairs, 128.4ms
video 1/1 (frame 10/489) /content/webcam_ada_orang.mp4: 384x6